In [ ]:
# Import libraries and initialize Earth Engine
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

ee.Authenticate()
ee.Initialize(project='qsair-463811')

In [ ]:
# Define study area 
study_area = ee.Geometry.Rectangle([31.30, 1.60, 32.20, 2.50])  # Murchison Falls bounds

In [ ]:
# Load and filter Sentinel-2 collection
s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(study_area)
                 .filterDate('2022-05-01', '2022-09-30')  # Growing season dates
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

print(f"Number of images in collection: {s2_collection.size().getInfo()}")

In [ ]:
# Define and apply NDVI calculation function
def addNDVI(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)

# Apply function to the collection
s2_collection_with_ndvi = s2_collection.map(addNDVI)

In [ ]:
# Calculate maximum NDVI composite
max_ndvi = s2_collection_with_ndvi.select('NDVI').max()

# Visualization parameters for NDVI
ndvi_vis_params = {
    'min': 0.1,
    'max': 0.8,
    'palette': ['red', 'yellow', 'green']
}

In [ ]:
# Calculate mean maximum NDVI for study area
mean_ndvi_study_area = max_ndvi.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=study_area,
    scale=10,
    maxPixels=1e9
).get('NDVI').getInfo()

print(f'Mean maximum NDVI for study area: {mean_ndvi_study_area:.3f}')

In [ ]:
# Set up a specific point for time series analysis
specific_point = ee.Geometry.Point([31.645, 2.218])  # Longitude, Latitude

In [ ]:
# Extract NDVI time series for the point
ndvi_time_series = s2_collection_with_ndvi.select('NDVI').getRegion(
    geometry=specific_point,
    scale=10
).getInfo()

# Convert to pandas DataFrame
ndvi_df = pd.DataFrame(ndvi_time_series[1:], columns=ndvi_time_series[0])
ndvi_df['time'] = pd.to_datetime(ndvi_df['time'], unit='ms')

# Display first few rows
ndvi_df.head()

In [ ]:
# Plot the NDVI time series
plt.figure(figsize=(12, 6))
plt.plot(ndvi_df['time'], ndvi_df['NDVI'], marker='o', linestyle='-')
plt.xlabel('Date')
plt.ylabel('NDVI Value')
plt.title('NDVI Time Series for Specific Point')
plt.grid(True)
plt.show()

In [ ]:
# Create interactive map visualization
# Center map on study area
Map = geemap.Map(center=[2.20, 31.65], zoom=10)  # Centered on the park
# Add layers to the map
Map.addLayer(max_ndvi, ndvi_vis_params, 'Maximum NDVI')
Map.addLayer(study_area, {'color': 'blue'}, 'Study Area', False)
Map.addLayer(specific_point, {'color': 'yellow'}, 'Time Series Point')

# Set basemap and display
Map.setOptions('HYBRID')
Map